In [7]:
# symmetric encryption + MAC

# Encryption: (K, E, D) 
# MAC:        (K, T, V) 

import secrets #for key generation token_bytes(?)
import hmac    #for implementing the standard MAC using SHA256

import hashlib # to make the key not recoverable

# K for encryption and MAC, different key for encryption and mac
def KeyGen():
    
    enc_key = secrets.token_bytes(32)  
    mac_key = secrets.token_bytes(32)
    return enc_key, mac_key


# While the key itself is random, the repeated use of key can give a hint from cipher text. 
# This disobeys the Kerckhoffs's Principle, thus we use initialization vector to randomize the key, while 
# including it in the cipher text.

def Encrypt(enc_key, text) :

    # generating vector to randomize the result
    initialized_vector = secrets.token_bytes(32)
    
    # while vector is randomized, it is revealed to the public so as the reciver can use this for decryption
    # Need to make it impossible for the adversary to recover the key, and it is okay to be invertible

    # Uses hash here, so we can get this value only when we know both enc_key and vector
    actual_key = hashlib.sha256(enc_key + initialized_vector).digest()
    
    # The limitation of OTP, the plain text can not be longer than key -> need to divide down to chunks and put it
    # in block cipher
    if len(text) > 32:
        raise ValueError("Message longer than 32")
    
    # equalizing the length of key with length of text for xor
    actual_key = actual_key[:len(text)]
    
    # Using XOR -> perfect secrecy by shannon unlike substitution,
    # EX) XYZ -> DFF not possible, substitution not perfect secrecy
    ciphertext = bytearray(text)
    for i in range(len(ciphertext)):
        ciphertext[i] ^= actual_key[i]
    
    # returning the concatenation of vector and ciphertext -> anyone can parse it from ciphertext and try using
    # it for decryption, but only with actual key can actually decrypt it
    return initialized_vector + ciphertext


def Decrypt(enc_key, public_dec):
    
    # parsing first 32 bit and second 32 bit from the decryption recieved
    initialized_vector = public_dec[:32]
    ciphertext = public_dec[32:]
    
    # regenerate the actual key concating enc_key and vector then hashing
    actual_key = hashlib.sha256(enc_key + initialized_vector).digest()
    actual_key = actual_key[:len(ciphertext)]
    
    # Now, do Xor operation with actual key
    message = bytes(c ^ k for c, k in zip(ciphertext, actual_key))
    
    return message


# Mac T(K,M) -> sigma
# the problem with the authentication key tag in symmetry encryption is that the reciever can
# fake recieving certain tag as they use the same key. While the asymetry encryption do not have 
# such problem as the encryption and decrytion key is different
def T(mac_key, message):

    # by importing hmac, we use the standardized mac using sha-256
    # sigma = HMAC(K, M)
    tag = hmac.new(mac_key, message, hashlib.sha256).digest()
    return tag



# Verigying Mac scheme, V algorithm: V(K, M, sigma) -> boolean

def V(mac_key, message, tag_recieved):
    # regenerate the expected 
    tag_compare = hmac.new(mac_key, message, hashlib.sha256).digest()
    
    # use inner function of hmac to compare the two tag, very convenient. The 0 or 1 will be returned
    return hmac.compare_digest(tag_recieved, tag_compare)


# finalize the whole thing by adding up encrytion and MAC

def AuthEnc(enc_key: bytes, mac_key: bytes, message: bytes) -> tuple[bytes, bytes]:
    
    # Encrypting for the first
    ciphertext = Encrypt(enc_key, message)
    
    # We MAC the ciphertext
    tag = T(mac_key, ciphertext)
    
    return ciphertext, tag


def AuthDec(enc_key, mac_key, ciphertext, tag):
    # checking if authentication is valid, using raise ValueError to catch the error using try later on
    if not V(mac_key, ciphertext, tag):
        raise ValueError("⊥ Authentication failed — message rejected")
    
    # When authentication valid, decrypt
    message = Decrypt(enc_key, ciphertext)
    
    return message

In [10]:
print("=== 1. Test correctness ====\n")

# generate keys seperately
enc_key, mac_key = KeyGen()
message = b"Attack Midway at dawn!"

# encrypt and authenticate using the AuthEnc
ciphertext, tag = AuthEnc(enc_key, mac_key, message)
# print origingal message, cipher text, and the tag
print(f"Original message:  {message}")
print(f"Ciphertext:        {ciphertext.hex()}")
print(f"Tag (σ):           {tag.hex()}\n")

# check authentication validity and decrypt using AuthDEC
recovered = AuthDec(enc_key, mac_key, ciphertext, tag)
print(f"Recovered message: {recovered}")
if recovered == message :
    print("The authentication encryption system works!  ✅\n")
else:
    print("The authentication encryption system do not work\n")

print("=== 2. Check what happens when implementing the wrong key ===\n")

not_enc_key, not_mac_key = KeyGen()
try:
    AuthDec(not_enc_key, not_mac_key, ciphertext, tag)
except ValueError as e:
    print(f"Wrong key correctly rejected: {e} ✅\n")


print("=== 3. Check what happens when ciphertext is converted===\n")

# Adversary converting the ciphertext
tampered = bytearray(ciphertext) # change into bytearray for xor operation
tampered[2] ^= 1  # changing one bit at the 3rd place
tampered = bytes(tampered) # change it back to bytes

# While authentication, as the message is converted, the Auth validation using SHA256 should fail
try:
    AuthDec(enc_key, mac_key, tampered, tag)
except ValueError as e:
    print(f"Tampering correctly detected: {e} ✅\n")

print("=== 4. Check the randomness of algorithm ===\n")

# As this is the random symmery encryption schema using the initialized vector generated randomly, the same message should be encrypted into 
# different cipher text every time it gets encrypted
C1, tag1 = AuthEnc(enc_key, mac_key, message)
C2, tag2 = AuthEnc(enc_key, mac_key, message)

# print out the cipher text of each
print(f"Ciphertext 1: {C1.hex()}")
print(f"Ciphertext 2: {C2.hex()}")
if C1 != C2 :
    print("\nDifferent ciphertext for same input  ✅")
else:
    print("\nThe encryption system is not random, need change")

=== 1. Test correctness ====

Original message:  b'Attack Midway at dawn!'
Ciphertext:        597b6e592b9f7a88bee6deb2fcf94623f160b58baa5a414a35e2ce90bcb893fc4f0c29e219aee3730c1670584d1b28d69c025c5e3801
Tag (σ):           d378df09eb3a17e4e3a9c18a2c1a78f35bb27f823efea950d502d5e42d1f3f74

Recovered message: b'Attack Midway at dawn!'
The authentication encryption system works!  ✅

=== 2. Check what happens when implementing the wrong key ===

Wrong key correctly rejected: ⊥ Authentication failed — message rejected ✅

=== 3. Check what happens when ciphertext is converted===

Tampering correctly detected: ⊥ Authentication failed — message rejected ✅

=== 4. Check the randomness of algorithm ===

Ciphertext 1: ff1033b0af6f5cd4dc61e0d6d6fea7cf38a85db68ac42a4dc19a068ff178aaa709221d4465a5fa370db1cd9eae2f6f3c621b0d9952d1
Ciphertext 2: ba5d4d11723341a8d5141bd5de40bb2c7e3915be28caceb56d016302e1f604be5b738a148a0506271c5506bf8fa1b1c7e412144fb8e1

Different ciphertext for same input  ✅
